# Checkpoint B -- the Part-2 recommender

This checkpoint closes **Part 2**. Over Units 7-13 you added meaning (word and book embeddings), a
learned two-tower model, cold-start features, a sparse + dense hybrid, sequence-aware paths, and a
whole-system view of who a recommender serves and what it does to its own data. Here you put the
Part-2 paths side by side, measure them the way a real system is measured -- accuracy **and** reach
**and** exposure -- and give one frozen recommender a single final grade on data it has never seen.

**How this checkpoint works -- freeze, then score** (the same discipline as Checkpoint A).

- **Questions 1-7 run on the `val` split only** (Question 5 runs on the *session* log's `val` split).
  You score each Part-2 path, read the cold-start and ethics readouts, then build the Part-2 blend,
  measure it on `val`, run a leave-one-path-out ablation, and **FREEZE** it at the end of Question 7.
- **Question 8 unseals the sealed `test` split of the main log for the second and last time in this
  book.** You score the FROZEN blend on `test` in one aggregate pass and interpret the result. You
  must not go back to Questions 1-7 to change anything after looking at `test`.

Fill the `None` / `"TODO"` slots, and answer each short-answer prompt in the string it names. Build
the generated recsys data (`recsys/data/generated/`) before you begin. Everything is measured at
`K = 10` with the 60 cold readers excluded, exactly as the frozen scoreboard does it.

**An honest warning before you start.** Earlier units taught you that "a blend always wins" is not a
law. This checkpoint tests whether you believe it. Report what your numbers say, including when the
blend you build *loses* on one axis.

## Setup -- the data and two cheap Part-1 paths

The cell below loads the main interaction log and fits two Part-1 paths you already know: **item-item
CF** (Unit 4) and **lexical / BM25** (Unit 3). They are *given*, not assessed on their own: each fits
in well under a second, and later questions reuse them -- as the warm-only and content baselines in
the cold-start readout (Question 4), as the members of the hybrid (Question 3) and of the Part-2 blend
(Questions 6-7), and as the feedback-loop path (Question 6). Run it once.

It also builds, from the **train** rows only, each reader's seen set and train-positive count per
book, the warm reader population (every non-cold reader with a train history), and each reader's
`val` positives. The `test` split stays SEALED until Question 8.

In [ ]:
import json
from collections import defaultdict

import bookrec
import numpy as np
import pandas as pd
from bookrec import (
    HybridRetrievalPath,
    ItemItemRetrievalPath,
    LastKItemItemPath,
    LexicalRetrievalPath,
    SemanticEmbeddingRetrievalPath,
    TransitionRetrievalPath,
    TwoTowerRetrievalPath,
    blend,
    book_embedding,
    exposure_gini,
    genre_spread_vs_history,
    head_ids_from_counts,
    load_glove_subset,
    load_train_sequences,
    long_tail_share,
    rank,
    reader_hits,
    run_blended_scoreboard,
    run_validation_scoreboard,
    simulate_feedback_loop,
    val_cohort,
)
from bookrec.lexical import tokenize

SEED = 0
K = 10      # every metric is reported at the top-10 cutoff
POOL = 30   # the per-path retrieval pool for blends (Unit 6)

data_dir = bookrec.generated_dir()
catalog = bookrec.load_catalog(data_dir / "catalog.csv.gz")
catalog_ids = sorted(catalog)        # the 2000-book catalog: the coverage / Gini denominator
keywords = bookrec.load_keywords(data_dir / "keywords.csv.gz")
interactions_path = data_dir / "interactions.csv.gz"

cold = json.loads((data_dir / "cold_partitions.json").read_text())
cold_readers = set(cold["cold_readers"])   # 60 readers no personalized path can serve

rows = pd.read_csv(interactions_path).to_dict("records")

# Train positives are each reader's `seen` set; `val` positives are the held-out relevant set.
seen_by = defaultdict(set)
val_by = defaultdict(set)
train_counts = defaultdict(int)      # book -> number of train positives
for row in rows:
    if row["label"] != 1:
        continue
    if row["split"] == "train":
        seen_by[row["reader_id"]].add(row["item_id"])
        train_counts[row["item_id"]] += 1
    elif row["split"] == "val":
        val_by[row["reader_id"]].add(row["item_id"])

# Every non-cold reader with a train history: the label-free population for exposure readouts.
warm_population = sorted(reader for reader in seen_by if reader not in cold_readers)
# Books with NO train positive: the cold-item denominator for Question 4.
zero_train_items = set(catalog_ids) - set(train_counts)

item_item = ItemItemRetrievalPath().fit(rows, catalog=catalog_ids)
lexical = LexicalRetrievalPath(keywords).fit(rows, catalog=catalog_ids)

{
    "books": len(catalog_ids),
    "warm_population": len(warm_population),
    "zero_train_items": len(zero_train_items),
    "fitted": [item_item.name, lexical.name],
}

## Question 1 -- Meaning, not words: semantic retrieval

Lexical retrieval (Unit 3) matches a reader's history to books that share *words*. Semantic retrieval
(Unit 7) matches *meaning*: every keyword becomes a pretrained **word embedding** (the committed GloVe
subset), a book becomes the L2-normalized mean of its keyword vectors (a **document embedding**), a
reader becomes the mean of the books they read, and retrieval returns the books with the highest
cosine (**embedding retrieval**).

1. Build one book's embedding by hand with `book_embedding(tokenize(keywords[book]), glove)` for
   `book = catalog_ids[0]`, and record its norm (it should be 1, up to rounding).
2. Fit a `SemanticEmbeddingRetrievalPath` and score it on `val` with `run_validation_scoreboard`
   (measured on this data: hit@10 about 0.102, *below* lexical's 0.158).

Then, in two or three sentences: why can the semantic path score below lexical on hit@10 and still be
worth keeping in a system? (Think about what it surfaces that lexical does not.)

In [ ]:
glove = None        # TODO: load_glove_subset()

sample_book = catalog_ids[0]
sample_embedding = None   # TODO: book_embedding(tokenize(keywords[sample_book]), glove)
sample_norm = None        # TODO: float(np.linalg.norm(sample_embedding))   -- about 1.0

semantic = None     # TODO: SemanticEmbeddingRetrievalPath(keywords, glove).fit(rows, catalog=catalog_ids)
semantic_val = None # TODO: run_validation_scoreboard(semantic, interactions_path,
                    #        catalog_ids=catalog_ids, k=K, cold_readers=cold_readers)
semantic_hit = None # TODO: semantic_val.hit_rate_at_k            (about 0.102)
lexical_hit = None  # TODO: the same scoreboard for `lexical`     (about 0.158)

semantic_reading = "TODO"  # 2-3 sentences: meaning-based matches reach books with DIFFERENT words
                           # (low overlap with lexical), so a weaker single path can still add
                           # reach and complementary candidates to a blend

## Question 2 -- A learned model: the two-tower

The two-tower (Unit 8) learns a **reader tower** and a **book tower** in PyTorch: a reader's score for
a book is the dot product of their two embeddings. It is trained with the **BPR loss** -- for a reader,
a book they read should outscore a sampled book they did not -- by mini-batch gradient descent with
weight decay (**neural training**).

Fit **one** `TwoTowerRetrievalPath(seed=SEED)` on the main log (about 15-25 s; fit it once and reuse
it for the rest of the checkpoint). Record the first and last epoch of its BPR loss history, and score
it on `val`. Then answer in two or three sentences: what does a *falling* BPR loss tell you, and what
does it *not* tell you about hit@10 on held-out readers?

In [ ]:
two_tower = None        # TODO: TwoTowerRetrievalPath(seed=SEED).fit(rows, catalog=catalog_ids)
                        #        -- ONE fit; reuse it in Questions 3, 4, 6, 7 and 8

first_loss = None       # TODO: two_tower.loss_history[0]
last_loss = None        # TODO: two_tower.loss_history[-1]
loss_fell = None        # TODO: bool - last_loss < first_loss

two_tower_val = None    # TODO: run_validation_scoreboard(two_tower, interactions_path,
                        #        catalog_ids=catalog_ids, k=K, cold_readers=cold_readers)
two_tower_hit = None    # TODO: two_tower_val.hit_rate_at_k       (about 0.340 -- the best single path)

bpr_reading = "TODO"  # 2-3 sentences: a falling BPR loss means the model ranks TRAIN positives above
                      # sampled negatives better; it says nothing directly about unseen val books --
                      # only the held-out scoreboard measures that

## Question 3 -- Sparse + dense: the hybrid

Unit 10's **hybrid retrieval** fuses a sparse path (BM25 / lexical) and a dense path (the two-tower):
each path retrieves a pool of 50 candidates, their scores are min-max normalized, and a weighted sum
(dense weight 0.7) orders the union. Build it from the paths you already fit -- **no new fit** -- with
`HybridRetrievalPath(lexical, two_tower, weight=0.7, pool=50).fit(rows)`.

Measure hit@10 and catalog coverage for the two-tower alone and for the hybrid, each as a single-path
blend through `run_blended_scoreboard` with `pool=K` (measured: two-tower about 0.340 / 0.177, hybrid
about 0.362 / 0.192). In two or three sentences: what did fusion buy here, and why is "fusion is a free
win" still the wrong lesson? (Recall what equal weights did in Unit 10.)

In [ ]:
def single_path_board(path):
    # A single-path blend is just that path; the blended scoreboard also reports coverage.
    return run_blended_scoreboard(
        {path.name: path}, interactions_path, catalog_ids=catalog_ids,
        k=K, pool=K, cold_readers=cold_readers,
    )


hybrid = None            # TODO: HybridRetrievalPath(lexical, two_tower, weight=0.7, pool=50).fit(rows)

two_tower_board = None   # TODO: single_path_board(two_tower)
hybrid_board = None      # TODO: single_path_board(hybrid)

two_tower_coverage = None  # TODO: two_tower_board.catalog_coverage   (about 0.177)
hybrid_hit = None          # TODO: hybrid_board.hit_rate_at_k         (about 0.362)
hybrid_coverage = None     # TODO: hybrid_board.catalog_coverage      (about 0.192)

hybrid_reading = "TODO"  # 2-3 sentences: a small lift on BOTH hit and coverage at this tuned,
                         # dense-heavy weight; the lift is weight-sensitive (equal weights lost to the
                         # two-tower alone in Unit 10), so fusion must be measured, not assumed

## Question 4 -- Cold start: who can recommend a book nobody has read?

A **cold item** is a book with no train positive. The setup cell built `zero_train_items`, the set of
catalog books no reader has a train positive for. Using **only paths you have already fit** -- semantic,
lexical, item-item and the two-tower; no new model -- measure two things per path:

1. **Cold reach:** over every reader in `warm_population`, retrieve that reader's top-10 (excluding
   their seen set) and count the *distinct* zero-train books that appear anywhere. This needs no
   held-out labels.
2. **Cold hit:** over the non-cold readers who have at least one zero-train book among their unseen
   `val` positives, the fraction whose top-10 contains one of those cold books.

Then, in two or three sentences: which kinds of path serve cold books and which surface exactly
zero, and *why* -- what does each kind of path need to know about a book before it can recommend it?
(Unit 9's feature tower is the learned path built to close this gap; you do not need to fit it here.)

In [ ]:
def cold_reach(path):
    # Distinct zero-train books in any warm reader's top-10 (train history only, no labels).
    surfaced = set()
    for reader in warm_population:
        seen = seen_by[reader]
        top = None   # TODO: rank(path.retrieve(reader, {"seen": seen}, K), n=K, exclude=seen)
        # TODO: add every top-10 item_id that is in zero_train_items to `surfaced`
    return len(surfaced)


def cold_hit(path):
    # hit@10 restricted to readers whose unseen val positives include a zero-train book.
    hits = []
    for reader in sorted(val_by):
        if reader in cold_readers:
            continue
        seen = seen_by[reader]
        cold_targets = None   # TODO: (val_by[reader] - seen) & zero_train_items
        # TODO: skip readers with no cold target; otherwise append 1.0 if the reader's top-10
        #       contains a cold target, else 0.0
    return (sum(hits) / len(hits) if hits else 0.0), len(hits)


cold_rows = []
for name, path in (("semantic", None), ("lexical", lexical), ("item-item", item_item),
                   ("two-tower", None)):   # TODO: put the fitted semantic and two-tower paths here
    reach = None             # TODO: cold_reach(path)
    hit, n_readers = None, None   # TODO: cold_hit(path)
    cold_rows.append({"path": name, "zero_train_surfaced": reach, "cold_hit@10": hit,
                      "cold_target_readers": n_readers})

cold_reading = "TODO"  # 2-3 sentences: content paths (semantic, lexical) place a cold book from its
                       # keywords and DO surface cold books; warm-only collaborative paths (item-item,
                       # two-tower) learn only from interactions, so a book with none surfaces 0

## Question 5 -- Order matters: sequence-aware paths (session log, `val` only)

Unit 12 used a **different log**: `sessions.csv.gz`, whose readers' reads come in a meaningful order
(series volumes, sessions). Everything in this question runs on that session log and on its `val`
split only; its numbers are **not comparable** with the main-log numbers elsewhere in this checkpoint.

Fit four cheap numpy paths on the session log:
- `ItemItemRetrievalPath()` -- the whole-history "bag" baseline;
- `LastKItemItemPath(3)` -- item-item queried with only the reader's last 3 distinct reads (recency);
- `TransitionRetrievalPath(5)` -- first-order item-to-item transitions from the last reads;
- `TransitionRetrievalPath(5, shuffle_seed=SHUFFLE_SEED, name="transition-shuffled")` -- the same
  path trained on each reader's history in a **shuffled order**, the control that destroys order but
  keeps every book.

Score each with `reader_hits` (per-reader hit@10 on the scoreboard's cohort; no `cold_readers` on the
session log) and compute two **paired** differences with their paired standard error
(`diffs.std(ddof=1) / sqrt(n)`): last-3 minus bag, and transition minus its shuffled control. Then, in
two or three sentences: what does each comparison show, and why is the shuffled control a stronger
piece of evidence for *order* than simply "the transition path scores well"?

In [ ]:
SHUFFLE_SEED = 20260930 + 1000   # the order-shuffled control seed (committed data seed + 1000)

sessions_path = data_dir / "sessions.csv.gz"           # the SESSION log -- different readers
session_rows = pd.read_csv(sessions_path).to_dict("records")
histories = load_train_sequences(sessions_path)        # each reader's ORDERED train positives

bag = None                   # TODO: ItemItemRetrievalPath().fit(session_rows, catalog=catalog_ids)
last_3 = None                # TODO: LastKItemItemPath(3).fit(session_rows, catalog=catalog_ids)
transition = None            # TODO: TransitionRetrievalPath(5).fit(session_rows, catalog=catalog_ids)
transition_shuffled = None   # TODO: TransitionRetrievalPath(5, shuffle_seed=SHUFFLE_SEED,
                             #        name="transition-shuffled").fit(session_rows, catalog=catalog_ids)

session_hits = {}
for label, path in (("bag", bag), ("last-3", last_3), ("transition", transition),
                    ("transition-shuffled", transition_shuffled)):
    session_hits[label] = None   # TODO: reader_hits(path, sessions_path, catalog_ids=catalog_ids, k=K)


def paired(label_a, label_b):
    # Mean per-reader hit difference (a - b) and its paired standard error.
    readers = sorted(session_hits[label_a])
    diffs = None   # TODO: np.array([session_hits[label_a][r] - session_hits[label_b][r] for r in readers])
    return None    # TODO: (float(diffs.mean()), float(diffs.std(ddof=1) / np.sqrt(len(diffs))))


recency_lift, recency_se = None, None         # TODO: paired("last-3", "bag")
order_gap, order_se = None, None              # TODO: paired("transition", "transition-shuffled")

sequence_reading = "TODO"  # 2-3 sentences: recency (last-3) beats the whole bag; the transition path
                           # clearly beats its OWN shuffled copy -- same books, same model, only order
                           # destroyed -- so the gap is evidence the order itself carries signal

## Question 6 -- Ethics: exposure, filter bubbles, and a feedback loop

Unit 13 asked who a recommender is good for and what it does to its own data. Answer three parts.
Every claim you write must come with one of your numbers.

**The Part-2 blend configuration** for this checkpoint is pinned: an **equal-weight** blend of
**item-item, two-tower, semantic and lexical**, each contributing a pool of `POOL = 30` candidates.
(You measure it on `val` and freeze it in Question 7; here you only look at what it *serves*.)

**(a) Exposure fairness.** For every reader in `warm_population`, compute the blend's top-10 with
`blend` + `rank` (the helper below), and the two-tower's own top-10 for comparison. For both, report
the **exposure Gini** over the full catalog (`exposure_gini`) and the **long-tail share** (the
fraction of slots that go to non-head books; the head is the top 10% of books by train count,
`head_ids_from_counts(train_counts, catalog_ids)`). These readouts use no held-out labels.

**(b) Filter bubbles.** With `genre_spread_vs_history(blend_lists, seen_by, catalog)`, compare each
reader's recommended genre entropy with the genre entropy of their own history. Report the mean of
each, and the fraction of readers whose recommendations are *narrower* than their history.

**(c) A feedback loop, with and without exploration.** Run `simulate_feedback_loop` twice on
item-item CF, for `rounds=3` (short, to keep the checkpoint fast) over the val cohort
(`val_cohort(rows, catalog_ids, cold_readers)`), `seed=SEED`, once with no mitigation and once with
`mitigation="explore"` (one random unseen book in the last slot). Pass `path_factory` as a function
that returns a *fresh, unfitted* `ItemItemRetrievalPath()` -- the loop refits it every round. Compare
round 0 with round 3 on coverage, Gini, genre entropy and clean hit.

Then, in four to six sentences: what do (a)-(c) show; why is no single number here "the fairness" of
the system; and why must the loop never let a simulated reader "accept" one of their own `val` (or
other held-out) books -- what would that do to the clean-hit column?

In [ ]:
PART2_PATHS = None   # TODO: {"item-item": item_item, "two-tower": two_tower,
                     #        "semantic": semantic, "lexical": lexical}
PART2_WEIGHTS = None # TODO: {name: 1.0 for name in PART2_PATHS}   -- equal weights


def blend_top_k(paths, weights, reader):
    # The blend's top-10 for one reader: per-path pools -> calibrated weighted union -> rank.
    seen = seen_by[reader]
    per_path = None   # TODO: {name: path.retrieve(reader, {"seen": seen}, POOL) for name, path in paths.items()}
    return None       # TODO: rank(blend(per_path, weights=weights), n=K, exclude=seen)


head_ids = None       # TODO: head_ids_from_counts(train_counts, catalog_ids)

# (a) exposure: what each recommender SERVES to the warm population (no labels needed)
blend_lists = None    # TODO: {reader: blend_top_k(PART2_PATHS, PART2_WEIGHTS, reader)
                      #        for reader in warm_population}
two_tower_lists = None  # TODO: {reader: rank(two_tower.retrieve(reader, {"seen": seen_by[reader]}, K),
                        #        n=K, exclude=seen_by[reader]) for reader in warm_population}

blend_gini = None          # TODO: exposure_gini(blend_lists, catalog_ids)
two_tower_gini = None      # TODO: exposure_gini(two_tower_lists, catalog_ids)
blend_long_tail = None     # TODO: long_tail_share(blend_lists, head_ids)
two_tower_long_tail = None # TODO: long_tail_share(two_tower_lists, head_ids)

# (b) filter bubbles: recommended genre spread vs each reader's own history
spread = None              # TODO: genre_spread_vs_history(blend_lists, seen_by, catalog)
mean_rec_entropy = None    # TODO: mean of the first value of each (recommended, history) pair
mean_history_entropy = None  # TODO: mean of the second value
narrower_fraction = None   # TODO: fraction of readers whose recommended entropy < history entropy

In [ ]:
# (c) two short feedback loops (R = 3) on item-item CF -- run each loop ONCE; about 10 s each.
cohort = None   # TODO: val_cohort(rows, catalog_ids, cold_readers)


def fresh_item_item():
    return None   # TODO: a fresh, UNFITTED ItemItemRetrievalPath() -- the loop calls fit itself


loop_plain = None     # TODO: simulate_feedback_loop(fresh_item_item, rows, cohort=cohort,
                      #        catalog=catalog, seed=SEED, rounds=3)
loop_explore = None   # TODO: the same call with mitigation="explore"

loop_rows = []
for label, loop in (("no mitigation", loop_plain), ("exploration slot", loop_explore)):
    for metrics in (loop[0], loop[-1]):   # round 0 and round 3
        loop_rows.append({
            "loop": label,
            "round": None,          # TODO: metrics.round
            "coverage": None,       # TODO: metrics.coverage
            "gini": None,           # TODO: metrics.gini
            "genre_entropy": None,  # TODO: metrics.genre_entropy
            "clean_hit": None,      # TODO: metrics.clean_hit
        })

ethics_reading = "TODO"  # 4-6 sentences, every claim with a number: (a) exposure concentration of
                         # the blend vs the two-tower; (b) narrower-than-history share; (c) the loop
                         # concentrates (coverage down, Gini up) and exploration partly restores it;
                         # each metric is ONE descriptive proxy, not "the" fairness; the leakage
                         # guard -- accepting a reader's own held-out book would hand the model the
                         # answer and make clean hit RISE for the wrong reason

## Question 7 -- Build the Part-2 blend on `val`, then FREEZE it

Now measure the pinned Part-2 blend from Question 6 -- equal weights over item-item, two-tower,
semantic and lexical, pool 30 -- on `val` with `run_blended_scoreboard`, and record its hit@10 and
catalog coverage. Then run a **leave-one-path-out ablation** over its four members: drop one path,
re-blend the other three at the same pool with equal weights, and record the signed change in hit@10
and in coverage. Report the hybrid from Question 3 as a **separate single path** for comparison, not
as a blend member (it already contains the two-tower, so blending both would double-count it).

Read your numbers honestly. On this data the plan's measurement of this exact blend was about **hit
0.298 / coverage 0.383**: it **loses** on hit@10 to its own two-tower member (0.340) and to the hybrid
(0.362), and dropping either content path (semantic or lexical) *raises* hit@10. What it buys is
**reach**: about twice the two-tower's catalog coverage (0.177) and the hybrid's (0.192). This is a
**hit <-> coverage trade**, not a win.

Then freeze. Put the members, weights and pool into `FROZEN_PATHS`, `FROZEN_WEIGHTS` and
`FROZEN_POOL`, and record the frozen `val` hit@10 and coverage in the cell marked `# FREEZE`. You
freeze the blend *as specified* -- do not swap in the two-tower or drop a member to chase hit@10, and
never revisit this choice after Question 8. In three or four sentences: compare the frozen blend with
the best single path on **both** axes, say what the ablation shows, and say who would prefer each
recommender.

In [ ]:
blend_val = None          # TODO: run_blended_scoreboard(PART2_PATHS, interactions_path,
                          #        catalog_ids=catalog_ids, k=K, pool=POOL,
                          #        weights=PART2_WEIGHTS, cold_readers=cold_readers)
blend_val_hit = None      # TODO: blend_val.hit_rate_at_k        (about 0.298)
blend_val_coverage = None # TODO: blend_val.catalog_coverage     (about 0.383)

ablation_rows = []
for dropped in ("item-item", "two-tower", "semantic", "lexical"):
    kept = None           # TODO: {name: path for name, path in PART2_PATHS.items() if name != dropped}
    board = None          # TODO: run_blended_scoreboard(kept, interactions_path, catalog_ids=catalog_ids,
                          #        k=K, pool=POOL, weights={name: 1.0 for name in kept},
                          #        cold_readers=cold_readers)
    ablation_rows.append({
        "dropped": dropped,
        "blend_hit@10": None,        # TODO: board.hit_rate_at_k
        "delta_hit": None,           # TODO: board.hit_rate_at_k - blend_val_hit   (signed)
        "delta_coverage": None,      # TODO: board.catalog_coverage - blend_val_coverage (signed)
    })
# Measured: -item-item 0.246, -two-tower 0.198, -semantic 0.310, -lexical 0.312.

comparison = None   # TODO: a small table (list of dicts) of hit@10 and coverage for the blend,
                    #        the two-tower alone (Question 3) and the hybrid (Question 3)
blend_beats_best_single_on_hit = None   # TODO: bool - blend_val_hit >= two_tower_hit

In [ ]:
# FREEZE -- every Part-2 decision is settled here, on val. Nothing above this cell may change after
# Question 8 is run.
FROZEN_PATHS = None        # TODO: the four members, exactly as in PART2_PATHS
FROZEN_WEIGHTS = None      # TODO: equal weights, exactly as in PART2_WEIGHTS
FROZEN_POOL = None         # TODO: POOL (30)
frozen_val_hit = None      # TODO: blend_val_hit
frozen_val_coverage = None # TODO: blend_val_coverage

blend_reading = "TODO"  # 3-4 HONEST sentences: the blend LOSES on hit@10 to the two-tower and the
                        # hybrid, but reaches about twice their catalog; the ablation shows the
                        # content paths cost hit and buy reach; neither recommender dominates --
                        # the choice depends on whether accuracy or catalog reach matters more

## Question 8 -- Unseal `test`: one honest final score

The main log's `test` split has been sealed since Checkpoint A. Now -- and only now, with every
decision frozen at the end of Question 7 -- score the FROZEN blend on it **exactly once**: one call to
`run_blended_scoreboard` with `FROZEN_PATHS`, `FROZEN_WEIGHTS`, `FROZEN_POOL`, the same cold-reader
exclusion, and `split="test"`. Write that call **once**, at the top level of the cell -- not in a loop,
a function or a condition -- and run the cell once. Report the test hit@10, the test catalog coverage
and the number of readers scored.

For the exposure Gini, think before you compute: the blend's top-10 for a reader depends only on that
reader's **train** history, never on which held-out split is being graded. So the frozen blend's
exposure Gini over the warm population is the number you already measured in Question 6(a) --
unsealing `test` cannot change it. Report it alongside the test numbers.

Then write three or four sentences of honest interpretation:
- how do the `test` numbers compare with the frozen `val` numbers (about 0.298 / 0.383)?
- how do they compare with Checkpoint A's frozen Part-1 blend, which scored about **0.306 hit@10 /
  0.338 coverage on `test`** -- did Part 2's extra machinery buy accuracy, reach, or neither, *for
  this blend*? (Remember the two-tower and the hybrid you did not freeze.)
- why must you not go back now and freeze the two-tower instead, even though Question 7 says it has
  the higher `val` hit@10?

In [ ]:
# The authorized one-shot unseal of the sealed main-log holdout. Run this cell ONCE, and do not return
# to Questions 1-7 to change anything after reading it.
blend_test = None   # TODO: run_blended_scoreboard(FROZEN_PATHS, interactions_path,
                       #        catalog_ids=catalog_ids, k=K, pool=FROZEN_POOL,
                       #        weights=FROZEN_WEIGHTS, cold_readers=cold_readers, split="test")
                       #        -- written ONCE, at the top level of this cell
test_hit = None       # TODO: blend_test.hit_rate_at_k
test_coverage = None  # TODO: blend_test.catalog_coverage
test_readers = None   # TODO: blend_test.readers

frozen_gini = None       # TODO: blend_gini from Question 6(a) -- label-free, unchanged by the unseal

final_report = None      # TODO: {"val": (frozen_val_hit, frozen_val_coverage),
                         #        "unsealed": (test_hit, test_coverage, test_readers),
                         #        "exposure_gini": frozen_gini}

final_interpretation = "TODO"  # 3-4 sentences: val vs held-out (close? over several hundred readers);
                               # vs Checkpoint A's ~0.306 / ~0.338 -- this blend trades hit for reach;
                               # swapping in the two-tower NOW would be tuning on the holdout, turning
                               # it into a second validation set